# Project Nexus — Live NVIDIA CUDA + vLLM Verification (Google Colab)

This notebook verifies **Project Nexus** end-to-end against a real **vLLM (NVIDIA CUDA)** OpenAI-compatible inference server on a Google Colab GPU runtime (T4, L4, or A100).

**Before running:**
1. In Colab, select **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU** (L4/A100 also work).
2. Run cells top-to-bottom.
3. Copy the final JSON summary output to record verified NVIDIA CUDA versions, GPU model, and latency metrics.

> **Scope note:** Google Colab provides NVIDIA GPUs only. Running this notebook verifies the **NVIDIA CUDA** path, not **AMD ROCm**.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# 1. Fail fast if Colab runtime is CPU-only
assert shutil.which("nvidia-smi"), "No NVIDIA GPU detected! Switch Colab runtime to GPU (L4/T4/A100)."
subprocess.run(["nvidia-smi"], check=True)

# 2. Clone or update Project Nexus
repo_dir = Path("/content/project-nexus")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/zeenat28-ui/project-nexus.git", str(repo_dir)],
        check=True,
    )

# 3. Install Project Nexus and vLLM (CUDA)
subprocess.run(["pip", "install", "-q", "-e", f"{repo_dir}[otel]"], check=True)
subprocess.run(["pip", "install", "-q", "vllm"], check=True)
print("Installed project-nexus and vllm successfully.")

In [ ]:
import json
import subprocess

gpu_proc = subprocess.run(
    ["nexus", "detect-gpu", "--json"],
    capture_output=True,
    text=True,
    check=True,
)
gpu_report = json.loads(gpu_proc.stdout)
print(json.dumps(gpu_report, indent=2))
assert gpu_report["nvidia"]["available"] is True, "Expected NVIDIA GPU to be detected by nexus detect-gpu"

In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # Apache-2.0, ungated, fits comfortably on T4/L4/A100
VLLM_PORT = 8001
vllm_log_path = Path("/tmp/vllm-server.log")
vllm_log_file = vllm_log_path.open("w", encoding="utf-8")

vllm_cmd = [
    sys.executable,
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--host",
    "127.0.0.1",
    "--port",
    str(VLLM_PORT),
    "--model",
    MODEL_ID,
    "--dtype",
    "half",
    "--max-model-len",
    "2048",
    "--gpu-memory-utilization",
    "0.80",
    "--enforce-eager",
]
print("Starting vLLM:", " ".join(vllm_cmd))
vllm_proc = subprocess.Popen(vllm_cmd, stdout=vllm_log_file, stderr=subprocess.STDOUT)

deadline = time.time() + 300
ready = False
while time.time() < deadline:
    if vllm_proc.poll() is not None:
        vllm_log_file.flush()
        print(vllm_log_path.read_text(encoding="utf-8")[-4000:])
        raise RuntimeError(f"vLLM exited early with code {vllm_proc.returncode}")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{VLLM_PORT}/health", timeout=2) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(3)

if not ready:
    vllm_log_file.flush()
    print(vllm_log_path.read_text(encoding="utf-8")[-4000:])
    raise TimeoutError("Timed out waiting for vLLM /health")

print(f"vLLM CUDA server is healthy on http://127.0.0.1:{VLLM_PORT} serving {MODEL_ID}")

In [ ]:
import json
import subprocess
from pathlib import Path

colab_cfg_path = Path("/tmp/nexus-colab-cuda.yaml")
colab_cfg_path.write_text(
    f"""
server:
  host: "127.0.0.1"
  port: 8000
  shutdown_drain_seconds: 5.0

backend:
  type: "openai_compatible"
  base_url: "http://127.0.0.1:{VLLM_PORT}"
  model: "{MODEL_ID}"
  api_style: "chat_completions"
  health_path: "/health"
  connect_timeout_seconds: 5.0
  read_timeout_seconds: 60.0

limits:
  request_timeout_seconds: 65.0
  health_check_timeout_seconds: 5.0
  max_concurrent_requests: 32
  rate_limit_requests_per_minute: 120
  rate_limit_burst: 20

metrics:
  mode: "separate"
  host: "127.0.0.1"
  port: 9090
""",
    encoding="utf-8",
)

verify_proc = subprocess.run(
    ["nexus", "verify-backend", "--config", str(colab_cfg_path), "--iterations", "5"],
    capture_output=True,
    text=True,
    check=False,
)
print(verify_proc.stdout)
if verify_proc.stderr:
    print("STDERR:", verify_proc.stderr)
assert verify_proc.returncode == 0, "nexus verify-backend failed!"
verify_report = json.loads(verify_proc.stdout)

In [ ]:
import json
import os
import secrets
import subprocess
import time
import urllib.error
import urllib.request

gw_token = secrets.token_urlsafe(24)
metrics_token = secrets.token_urlsafe(24)
env = os.environ.copy()
env["NEXUS_AUTH_API_KEY"] = gw_token
env["NEXUS_METRICS_BEARER_TOKEN"] = metrics_token

gw_proc = subprocess.Popen(
    ["nexus", "serve", "--config", str(colab_cfg_path)],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

gateway_checks = {}
try:
    for _ in range(40):
        try:
            with urllib.request.urlopen("http://127.0.0.1:8000/healthz", timeout=2) as r:
                assert r.status == 200
                break
        except Exception:
            time.sleep(0.25)
    else:
        raise RuntimeError("Project Nexus gateway failed to start")

    # 1. Readiness probe
    with urllib.request.urlopen("http://127.0.0.1:8000/readyz", timeout=5) as r:
        assert r.status == 200
        gateway_checks["readyz"] = json.loads(r.read())["status"]

    # 2. Auth enforcement check
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=5)
        raise AssertionError("Expected 401 without bearer token")
    except urllib.error.HTTPError as exc:
        assert exc.code == 401
        gateway_checks["auth_enforced"] = True

    # 3. Authenticated /v1/generate (non-streaming)
    gen_req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/generate",
        data=json.dumps(
            {"prompt": "Say 'CUDA verified' in two words.", "max_tokens": 24, "temperature": 0.0}
        ).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {gw_token}",
            "X-Request-ID": "colab-cuda-gen-1",
        },
        method="POST",
    )
    with urllib.request.urlopen(gen_req, timeout=30) as r:
        assert r.status == 200
        gen_data = json.loads(r.read())
        assert gen_data["output"].strip()
        gateway_checks["generate_latency_ms"] = gen_data["latency_ms"]
        gateway_checks["generate_output_preview"] = gen_data["output"][:80]

    # 4. Authenticated /v1/chat/completions (SSE streaming)
    chat_req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions",
        data=json.dumps(
            {
                "messages": [{"role": "user", "content": "Count from 1 to 5."}],
                "max_tokens": 32,
                "temperature": 0.0,
                "stream": True,
            }
        ).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {gw_token}",
            "X-Request-ID": "colab-cuda-sse-1",
        },
        method="POST",
    )
    with urllib.request.urlopen(chat_req, timeout=30) as r:
        assert r.status == 200
        assert r.headers["Content-Type"].startswith("text/event-stream")
        sse_text = r.read().decode("utf-8")
        assert "chat.completion.chunk" in sse_text
        assert "data: [DONE]" in sse_text
        gateway_checks["chat_sse_passed"] = True

    # 5. Metrics security: 404 on :8000, 200 with bearer token on :9090
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/metrics", timeout=5)
        raise AssertionError("Expected 404 on public /metrics")
    except urllib.error.HTTPError as exc:
        assert exc.code == 404

    metrics_req = urllib.request.Request(
        "http://127.0.0.1:9090/metrics",
        headers={"Authorization": f"Bearer {metrics_token}"},
    )
    with urllib.request.urlopen(metrics_req, timeout=5) as r:
        assert r.status == 200
        metrics_body = r.read().decode("utf-8")
        assert "nexus_requests_total" in metrics_body
        assert "CUDA verified" not in metrics_body
        assert "colab-cuda-gen-1" not in metrics_body
        gateway_checks["metrics_verified"] = True

    # 6. Live concurrent SSE load test through gateway -> vLLM CUDA
    lt_env = env.copy()
    lt_env["NEXUS_LOADTEST_API_KEY"] = gw_token
    lt_proc = subprocess.run(
        [
            "nexus",
            "load-test",
            "--url",
            "http://127.0.0.1:8000",
            "--endpoint",
            "/v1/chat/completions",
            "--requests",
            "10",
            "--concurrency",
            "2",
            "--max-tokens",
            "24",
            "--stream",
        ],
        env=lt_env,
        capture_output=True,
        text=True,
        check=True,
    )
    gateway_checks["concurrent_load_test"] = json.loads(lt_proc.stdout)
finally:
    gw_proc.terminate()
    gw_proc.wait(timeout=10)

print("Live gateway E2E checks passed:", json.dumps(gateway_checks, indent=2))

In [ ]:
import json
import torch
import vllm

# Shut down background vLLM server cleanly
if "vllm_proc" in globals() and vllm_proc.poll() is None:
    vllm_proc.terminate()
    vllm_proc.wait(timeout=15)
if "vllm_log_file" in globals():
    vllm_log_file.close()

summary = {
    "environment": {
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "vllm_version": vllm.__version__,
        "model": MODEL_ID,
    },
    "gpu": gpu_report["nvidia"],
    "verify_backend": {
        "overall_passed": verify_report["overall_passed"],
        "checks": verify_report["checks"],
        "latency_summary": verify_report["latency_summary"],
    },
    "gateway_e2e": gateway_checks,
}
print("=== COPY THIS VERIFICATION SUMMARY ===")
print(json.dumps(summary, indent=2))